In [1]:
# %% [markdown]
# # LR GW ~ P (6 predictores de memoria), con 8 series de precipitación por pozo
# - Una serie de P por nivel HydroBASINS (05 a 12): cuenca del pozo + aguas arriba.
# - Solo dos combinaciones: tendencia vs tendencia (`gw_an ~ pr_an`) y sin tendencia vs sin tendencia
#   (`gw_an_dt ~ pr_an_dt`). Se elimina `gw_an ~ pr_an_dt`.
# - Sin ET. Sin separación train/all: el modelo se ajusta con todo el período y hay un solo R².

# %%
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.append(str(Path('..').resolve()))
from fun_LR_hydro_memory import make_lag_ranges, rolling_predictors

# === Parámetros ===
data_dir = Path('../../data/ms_data/')
out_results = Path('csv')
out_results.mkdir(parents=True, exist_ok=True)

levels = range(7, 12)          # niveles HydroBASINS
n_pred = 6
min_obs = 100
start, end = '1960-01-01', '2024-12-31'

lag_configs = [
    {"incr_type": 1, "lag_increase": 4},
    {"incr_type": 2, "lag_increase": 4},
    {"incr_type": 3, "lag_increase": 2},
]

# Opcionales (pesan mucho si se activan con 8 niveles)
save_predictions = False       # guarda la simulación de cada modelo (millones de filas)
export_predictors = False      # guarda los 6 predictores por nivel para el lag type elegido
selected_lag_type = 'type2_incr4'

combinations = [               # (target, predictor, etiqueta)
    ('gw_an',    'pr_an',    'GW_an=linreg(P_an)'),
    ('gw_an_dt', 'pr_an_dt', 'GW_an_dt=linreg(P_an_dt)'),
]

# %% [markdown]
# ## Datos: anomalías mensuales generadas por `wells_pr_hybas.py`

# %%
full_months = pd.date_range(start=start, end=end, freq='MS')


def load_wide(path):
    df = pd.read_csv(path, parse_dates=['date']).set_index('date')
    df.columns = df.columns.astype(str)
    return df.reindex(full_months)


gw_an_all = load_wide(data_dir / 'wells_gw_an.csv')
pr_an_lev = {l: load_wide(data_dir / f'wells_pr_an_lev{l:02d}.csv') for l in levels}
meta = {l: pd.read_csv(data_dir / f'wells_meta_lev{l:02d}.csv', dtype={'cr2sub_id': str})
           .drop_duplicates('cr2sub_id').set_index('cr2sub_id') for l in levels}


# %% [markdown]
# ## Funciones auxiliares

# %%
def deseason(s):
    """Anomalía mensual respecto al período start-end (idéntico a recalcularla desde la serie original)."""
    return s - s.groupby(s.index.month).transform('mean')


def detrend(s):
    x = np.arange(len(s))
    y = s.values
    m = ~np.isnan(y)
    if m.sum() < 2:
        return pd.Series(np.nan, index=s.index)
    c = np.polyfit(x[m], y[m], deg=1)
    return pd.Series(y - np.polyval(c, x), index=s.index)


def r2_corr(y, yhat):
    if np.std(y) == 0 or np.std(yhat) == 0:
        return np.nan
    return np.corrcoef(y, yhat)[0, 1] ** 2


def fit_lr(y, X):
    """Ajusta con todas las filas completas. Devuelve R² para los modelos con 1..n predictores."""
    d = pd.concat([y, X], axis=1).dropna()
    if len(d) < min_obs:
        return None
    yv = d.iloc[:, 0].values
    Xv = d.iloc[:, 1:].values
    x_mu, y_mu = Xv.mean(axis=0), yv.mean()
    Xc, yc = Xv - x_mu, yv - y_mu

    out = {}
    for k in range(1, Xv.shape[1] + 1):
        coef, *_ = np.linalg.lstsq(Xc[:, :k], yc, rcond=None)
        out[f'r2_pred{k}'] = r2_corr(yv, Xc[:, :k] @ coef + y_mu)
        if k == Xv.shape[1]:
            out['r2'] = out[f'r2_pred{k}']
            out['has_neg_coef'] = bool((coef < 0).any())
            Xp = X.dropna()
            out['pred_series'] = pd.Series((Xp.values - x_mu) @ coef + y_mu, index=Xp.index)
    out['n_obs'] = len(d)
    return out


# %% [markdown]
# ## Bucle: pozo × nivel × configuración de lags × combinación

# %%
rows = []
pred_file = out_results / 'gw_sim_LR_8basins.csv'
first_write = True
store = {}   # (nivel, pred k, tipo) -> {pozo: serie}

for cod in gw_an_all.columns:
    gw = gw_an_all[cod]
    if gw.notna().sum() < min_obs:
        continue

    gw_an = deseason(gw)
    targets = {'gw_an': gw_an, 'gw_an_dt': detrend(gw_an)}
    preds_well = []

    for lvl in levels:
        if cod not in pr_an_lev[lvl].columns or cod not in meta[lvl].index:
            continue
        pr = pr_an_lev[lvl][cod]
        if pr.notna().sum() < min_obs:
            continue
        pr_an = deseason(pr)
        pr_an_dt = detrend(pr_an)

        for cfg in lag_configs:
            lag_label = f"type{cfg['incr_type']}_incr{cfg['lag_increase']}"
            lag_ranges, widths, incr_name = make_lag_ranges(
                lag_increase=cfg['lag_increase'], n_windows=n_pred, incr_type=cfg['incr_type'])

            preds = {
                'pr_an':    rolling_predictors(pr_an,    lag_ranges, standardize=True),
                'pr_an_dt': rolling_predictors(pr_an_dt, lag_ranges, standardize=True),
            }

            if export_predictors and lag_label == selected_lag_type:
                for k, col in enumerate(preds['pr_an_dt'].columns, start=1):
                    store.setdefault((lvl, k, 'an_dt'), {})[cod] = preds['pr_an_dt'][col]
                    store.setdefault((lvl, k, 'an'), {})[cod] = preds['pr_an'][col]

            for target, pname, label in combinations:
                res = fit_lr(targets[target], preds[pname])
                if res is None:
                    continue
                sim = res.pop('pred_series')
                rows.append({
                    'well_id': cod, 'nivel': lvl,
                    'HYBAS_ID': meta[lvl].loc[cod, 'HYBAS_ID'],
                    'UP_AREA': meta[lvl].loc[cod, 'UP_AREA'],
                    'target': target, 'predictor': pname, 'lr_label': label,
                    'lag_type': lag_label, 'lag_incr_rule': incr_name,
                    'lag_incr': cfg['lag_increase'], 'lag_ranges': str(lag_ranges),
                    **res,
                })
                if save_predictions:
                    preds_well.append(pd.DataFrame({
                        'date': targets[target].index, 'well_id': cod, 'nivel': lvl,
                        'target': target, 'predictor': pname, 'lag_type': lag_label,
                        'target_val': targets[target].values,
                        'pred': sim.reindex(targets[target].index).values}))

    if save_predictions and preds_well:
        pd.concat(preds_well).to_csv(pred_file, mode='w' if first_write else 'a',
                                     header=first_write, index=False)
        first_write = False

r2_all = pd.DataFrame(rows)
r2_all.to_csv(out_results / 'r2_LR_8basins.csv', index=False)

# Mejor nivel por pozo, combinación y configuración de lags
idx = r2_all.groupby(['well_id', 'target', 'predictor', 'lag_type'])['r2'].idxmax().dropna()
best = r2_all.loc[idx]
best.to_csv(out_results / 'r2_LR_8basins_best_level.csv', index=False)

if export_predictors:
    for (lvl, k, kind), d in store.items():
        df = pd.DataFrame(d).sort_index().rename_axis('date').reset_index()
        df.to_csv(data_dir / f"wells_pr_{kind}_lev{lvl:02d}_pred{k}_{selected_lag_type}.csv", index=False)

print(f"{r2_all.well_id.nunique()} pozos, {len(r2_all)} modelos")
print("\nLR models Finished! \nFinito! \nTerminado!\n")

504 pozos, 15110 modelos

LR models Finished! 
Finito! 
Terminado!

